# HPPCS[04] Multimodal Medical Assistant on Google Colab

Your PC has no GPU, so MedGemma vision is too slow locally. Colab gives a free **T4 GPU**.

**Before you start:** `Runtime` → `Change runtime type` → Hardware accelerator → **T4 GPU** → Save.

This notebook installs Ollama on Colab, pulls `medgemma:4b` and `llama3.2:3b`, then runs the same `main.py` as on your laptop.

Educational prototype only. Not for real clinical use.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

If that command fails, you are on CPU. Change the runtime to GPU and re-run from the top.

## 2. Install Ollama and start the server

The project talks to Ollama at `http://127.0.0.1:11434`. On Colab that is this virtual machine, not your laptop.

In [ ]:
import os, subprocess, time, requests

if not os.path.exists("/usr/local/bin/ollama"):
    !curl -fsSL https://ollama.com/install.sh | sh

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "ab"),
    stderr=subprocess.STDOUT,
)

for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("Ollama is running")
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. Check /tmp/ollama.log")

## 3. Get the project code

**Preferred if you have not pushed the latest rewrite to GitHub:** zip the `Codebase` folder on your PC, upload it here, then unzip.

If GitHub already has this rewrite, you can clone instead.

In [ ]:
from google.colab import files
import zipfile, os, shutil

# Option A: upload Codebase.zip from your PC (uncomment these 3 lines)
# uploaded = files.upload()
# zip_name = next(iter(uploaded))
# zipfile.ZipFile(zip_name).extractall("/content")

# Option B: clone GitHub (only if that repo has the LangGraph rewrite)
!git clone --depth 1 https://github.com/sumanchatterjeecs2010/capstone_iit.git /content/capstone_iit

if os.path.isdir("/content/capstone_iit/Codebase"):
    os.chdir("/content/capstone_iit/Codebase")
elif os.path.isdir("/content/Codebase"):
    os.chdir("/content/Codebase")
else:
    raise FileNotFoundError("Could not find Codebase/. Upload a zip or fix the clone path.")

print("Working directory:", os.getcwd())
print(os.listdir("."))

## 4. Python packages and models

`medgemma:4b` is about 3.3 GB. `llama3.2:3b` is about 2 GB. On a T4 they should be loaded **one at a time** (the code already unloads between steps).

In [ ]:
!pip -q install -r requirements.txt
!ollama pull medgemma:4b
!ollama pull llama3.2:3b
!ollama list

## 5. Run the assistant

This downloads the five public teaching images (if needed), then writes `conversation_01.json` … `conversation_05.json` and `evaluation_summary.json`.

On a T4 this often takes **10–25 minutes**, not hours.

In [ ]:
!python main.py

## 6. Download the outputs to your PC

In [ ]:
from google.colab import files
import glob

for path in sorted(glob.glob("conversation_*.json") + glob.glob("evaluation_summary.json")):
    print("Downloading", path)
    files.download(path)